## Regole di utilizzo monopattini elettrici
### Accesso al servizio
Tutti i giorni della settimana dalle ore 0.00 alle ore 24.00

### Tariffe
Le tariffe di utilizzo dei monopattini sono diverse a seconda dell’operatore che li fornisce:

- operatore **Bit Mobility Srl** (monopattini colore blu):
    - lo **sblocco** del mezzo costa **0,77 euro** + IVA
    - il **costo al minuto** è pari a **0,115 euro** + IVA
    - durante l’anno scolastico, nelle fasce orarie **dalle 7.15 alle 8.30** e **dalle 12.30 alle 13.30**, lo **sblocco** del mezzo è **gratuito** e il costo al minuto è pari a **0,057 euro** + IVA
    - nel caso di utilizzo degli **stalli associati alle riduzioni tariffarie** il costo di sblocco è pari a **0,39 euro** + IVA e il costo al minuto a **0,057 euro** + IVA

#
- operatore **Vento Mobility Srl** (monopattini colore celeste):
    - lo **sblocco** del mezzo è **gratuito**
    - il **costo al minuto** è pari a **0,195 euro** + IVA
    - durante l’anno scolastico l’utilizzo è **gratuito per i primi 30 minuti** di utilizzo nella fascia oraria **dalle 12.30 alle 13.30**
    - gli utenti che parcheggeranno presso i siti indicati dal Comune godranno di una **riduzione tariffaria pari ad € 1,00 per ogni parcheggio**.

#
Nella directory `data/static` è possibile trovare tutti i dati inerenti a:
- zone ZTL-P
- zone di parcheggio secondo l'ultima ordinanza
- aree proibite
- aree dove è possibile circolare con i monopattini 

In [95]:
import pandas as pd
import geopandas as gpd
import datetime as dt
from shapely.geometry import Point, Polygon
import geopy.distance

In [96]:
data = pd.read_parquet('data/trips_pointv3_cleaned.parquet')
data = gpd.GeoDataFrame(data, crs='EPSG:4326', geometry=gpd.points_from_xy(data.point_longitude, data.point_latitude))
data.head(3)

,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,date,unique_id,geometry
0,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:46:54+00:00,46.052974,11.120415,13,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07,POINT (11.12041 46.05297)
1,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:04+00:00,46.052951,11.120469,14,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07,POINT (11.12047 46.05295)
2,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:15+00:00,46.052905,11.120811,15,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07,POINT (11.12081 46.05291)


In [97]:
data.operator.unique()

array(['VENTO', 'BIT'], dtype=object)

In [98]:
area = gpd.read_file("data/static/punti-sosta-monopattini-con-ordinanza.shp") # area di stop associata alle riduzioni tariffarie
area = area.to_crs("EPSG:4326") # change reference system

In [99]:
def costo_trip(operator, time_start, time_end, stop_pt):
    
    # trip duration
    trip_time = time_end - time_start
    minut = trip_time.components.minutes + (trip_time.components.hours)*60
    d = time_start.date()

    # check se si ferma dentro aree di sosta a tariffa ridotta
    inside = False
    for ptsosta in range(len(area)):
        dist = geopy.distance.geodesic((stop_pt.x, stop_pt.y),(area.geometry[ptsosta].x, area.geometry[ptsosta].y))
        if dist.meters <= 3: # 3 metri
            inside = True
            break

    if operator == 'VENTO':
        # lo sblocco del mezzo è **gratuito**
        # il **costo al minuto** è pari a **0,195 euro** 
        if d.month in [9, 10, 11, 12, 1, 2, 3, 4, 5, 6]: # durante l’anno scolastico 
            if (time_start >= pd.Timestamp(str(d)+' 12:30:00+00:00') and time_start <= pd.Timestamp(str(d)+' 13:30:00+00:00')):
                # l’utilizzo è **gratuito per i primi 30 minuti** di utilizzo nella fascia oraria **dalle 12.30 alle 13.30**
                if trip_time <= pd.Timedelta('0 days 00:30:00'):
                    return 0
                else:
                    time_to_pay = trip_time - pd.Timedelta('0 days 00:30:00')
                    time_to_pay = time_to_pay.components.minutes + (time_to_pay.components.hours)*60
                    costo = time_to_pay*0.195
            else:
                costo = minut * 0.195
        else:
            costo = minut * 0.195
        # gli utenti che parcheggeranno presso i siti indicati dal Comune 
        # godranno di una **riduzione tariffaria pari ad € 1,00 per ogni parcheggio**
        if inside == True and costo >= 1: 
            return costo - 1
        elif inside == True and costo < 1:
            return 0
        else:
            return costo

    elif operator == 'BIT':
        if d.month in [9, 10, 11, 12, 1, 2, 3, 4, 5, 6]: # durante l’anno scolastico
            # nelle fasce orarie **dalle 7.15 alle 8.30** e **dalle 12.30 alle 13.30**
            # lo **sblocco** del mezzo è **gratuito** & costo al minuto è 0.057
            if time_start >= pd.Timestamp(str(d)+' 7:15:00+00:00') and time_start <= pd.Timestamp(str(d)+' 8:30:00+00:00'):
                # **dalle 7.15 alle 8.30** 
                if time_end <= pd.Timestamp(str(d)+' 8:30:00+00:00'): 
                    costo = 0 + (minut * 0.057)
                    return costo 
                else:
                    time_reduction = pd.Timestamp(str(d)+' 8:30:00+00:00') - time_start 
                    time_full_price = trip_time - time_reduction 
                    time_reduction = time_reduction.components.minutes + (time_reduction.components.hours)*60
                    time_full_price = time_full_price.components.minutes + (time_full_price.components.hours)*60
                    costo = (time_reduction * 0.057) + (time_full_price * 0.115)
                    return costo
            elif time_start >= pd.Timestamp(str(d)+' 12:30:00+00:00') and time_start <= pd.Timestamp(str(d)+' 13:30:00+00:00'):
                # **dalle 12.30 alle 13.30**
                if time_end <= pd.Timestamp(str(d)+' 13:30:00+00:00'): 
                    costo = 0 + (minut * 0.057)
                    return costo 
                else:
                    time_reduction = pd.Timestamp(str(d)+' 13:30:00+00:00') - time_start 
                    time_full_price = trip_time - time_reduction 
                    time_reduction = time_reduction.components.minutes + (time_reduction.components.hours)*60
                    time_full_price = time_full_price.components.minutes + (time_full_price.components.hours)*60
                    costo = (time_reduction * 0.057) + (time_full_price * 0.115)
                    return costo

            # inizio corsa al di fuori della finestra di riduzione ma overlap (morning)
            elif (time_start < pd.Timestamp(str(d)+' 7:15:00+00:00') and time_end > pd.Timestamp(str(d)+' 7:15:00+00:00')):
                # sblocco fuori dalla finestra gratuita (costo = 0.77)
                time_before_reduction = pd.Timestamp(str(d)+' 7:15:00+00:00') - time_start # timedelta
                if time_end < pd.Timestamp(str(d)+' 8:30:00+00:00'):
                    time_reduction = time_end - (time_start + time_before_reduction)
                    time_reduction = time_reduction.components.minutes + (time_reduction.components.hours)*60
                    time_before_reduction = time_before_reduction.components.minutes + (time_before_reduction.components.hours)*60
                    costo = (time_before_reduction*0.115) + (time_reduction*0.057) + 0.77
                    return costo
                else:
                    time_reduction = pd.Timestamp(str(d)+' 8:30:00+00:00') - (time_start + time_before_reduction)
                    time_after_reduction = time_end - pd.Timestamp(str(d)+' 8:30:00+00:00') 
                    time_reduction = time_reduction.components.minutes + (time_reduction.components.hours)*60
                    time_before_reduction = time_before_reduction.components.minutes + (time_before_reduction.components.hours)*60
                    time_after_reduction = time_after_reduction.components.minutes + (time_after_reduction.components.hours)*60
                    costo = (time_before_reduction*0.115) + (time_reduction*0.057) + (time_after_reduction*0.115) + 0.77
                    return costo
            # inizio corsa al di fuori della finestra di riduzione ma overlap (lunch time)
            elif (time_start < pd.Timestamp(str(d)+' 12:30:00+00:00') and time_end > pd.Timestamp(str(d)+' 12:30:00+00:00')):
                # sblocco fuori dalla finestra gratuita (costo = 0.77)
                time_before_reduction = pd.Timestamp(str(d)+' 12:30:00+00:00') - time_start # timedelta
                if time_end < pd.Timestamp(str(d)+' 13:30:00+00:00'):
                    time_reduction = time_end - (time_start + time_before_reduction)
                    time_reduction = time_reduction.components.minutes + (time_reduction.components.hours)*60
                    time_before_reduction = time_before_reduction.components.minutes + (time_before_reduction.components.hours)*60
                    costo = (time_before_reduction*0.115) + (time_reduction*0.057) + 0.77
                    return costo
                else:
                    time_reduction = pd.Timestamp(str(d)+' 13:30:00+00:00') - (time_start + time_before_reduction)
                    time_after_reduction = time_end - pd.Timestamp(str(d)+' 13:30:00+00:00') 
                    time_reduction = time_reduction.components.minutes + (time_reduction.components.hours)*60
                    time_before_reduction = time_before_reduction.components.minutes + (time_before_reduction.components.hours)*60
                    time_after_reduction = time_after_reduction.components.minutes + (time_after_reduction.components.hours)*60
                    costo = (time_before_reduction*0.115) + (time_reduction*0.057) + (time_after_reduction*0.115) + 0.77
                    return costo

            else:
                # full price
                # print('Sblocco 0,77 euro + 0.115 euro al minuto')
                costo = 0.77 + (minut * 0.115)
                return costo

        elif inside == True: # utilizzo degli **stalli associati alle riduzioni tariffarie** 
            # il costo di sblocco è pari a **0,39 euro** & il costo al minuto a **0,057 euro** 
            # print('Stalli associati alle riduzioni tariffarie')
            costo = 0.39 + (minut * 0.057)
            return costo 
        else:
            # full price
            # print('Sblocco 0,77 euro + 0.115 euro al minuto')
            costo = 0.77 + (minut * 0.115)
            return costo

    else:
        raise ValueError('Unexpected operator, got:', operator)

In [103]:
# costo_trip('BIT', pd.Timestamp('2022-02-07 12:29:00+00:00'), pd.Timestamp('2022-02-07 13:36:00+00:00'), Point(11.132729, 46.054454))

In [73]:
data = data.sort_values(by=['point_timestamp', 'unique_id'])

In [74]:
data.point_longitude = data.point_longitude.astype(float)
data.point_latitude = data.point_latitude.astype(float)

In [75]:
new_df = data.groupby('unique_id', as_index=False).aggregate({"point_longitude": lambda x: x.to_list(), "point_latitude": lambda x: x.to_list(), "point_timestamp":lambda x: x.to_list(), "operator": lambda x: list(set(x))})
new_df["coords"] = [ [[k,v] for k,v in zip(i,j)] for i,j in zip(new_df["point_longitude"], new_df["point_latitude"])]
new_df = new_df.drop(['point_latitude', 'point_longitude'], axis=1)
new_df

,unique_id,point_timestamp,operator,coords
0,00000000-0000-1000-8000-5b7d6a617b60_2021-06-10,"[2021-06-10 06:14:15+00:00, 2021-06-10 06:26:2...",[BIT],"[[11.119391, 46.071407], [11.115241, 46.060551]]"
1,00000000-0000-1000-8000-5b7d6a617b60_2021-07-31,"[2021-07-31 16:48:36+00:00, 2021-07-31 17:03:3...",[BIT],"[[11.123848, 46.075313], [11.124701, 46.060999]]"
2,00000000-0000-1000-8000-5b7d6a617b60_2021-08-05,"[2021-08-05 19:37:04+00:00, 2021-08-05 19:57:0...",[BIT],"[[11.124385, 46.051605], [11.115124, 46.090282]]"
3,00000000-0000-1000-8000-5b7d6a617b60_2021-09-28,"[2021-09-28 17:29:15+00:00, 2021-09-28 17:41:1...",[BIT],"[[11.123746, 46.062508], [11.115117, 46.090313]]"
4,00000000-0000-1000-8000-5b7d6a617b60_2021-10-01,"[2021-10-01 19:34:39+00:00, 2021-10-01 19:37:5...",[BIT],"[[11.125643, 46.066395], [11.119864, 46.069542]]"
...,...,...,...,...
88054,fff6b734bab3_2021-04-22,"[2021-04-22 17:10:48+00:00, 2021-04-22 17:10:5...",[VENTO],"[[11.119722, 46.086395], [11.119655, 46.086609]]"
88055,fff737605a72_2021-07-19,"[2021-07-19 11:37:02+00:00, 2021-07-19 11:37:0...",[VENTO],"[[11.120354, 46.067644], [11.120353, 46.067652..."
88056,fff77d5c39b5_2021-06-07,"[2021-06-07 16:25:20+00:00, 2021-06-07 16:25:2...",[VENTO],"[[11.124557, 46.07034], [11.124557, 46.070321]..."
88057,fff9fc12b926_2021-06-09,"[2021-06-09 06:33:31+00:00, 2021-06-09 06:39:3...",[VENTO],"[[11.123182, 46.078064], [11.123108, 46.077972]]"


Example:

In [76]:
new_df.coords[88058]

[[11.119815, 46.069758],
 [11.119815, 46.069761],
 [11.11981, 46.069779],
 [11.119806, 46.069788],
 [11.119798, 46.069785],
 [11.122523, 46.069284],
 [11.132729, 46.054454]]

In [77]:
new_df.point_timestamp[88058][-1] - new_df.point_timestamp[88058][0]

Timedelta('0 days 00:12:52')

In [78]:
new_df.operator[88058]

['VENTO']

In [79]:
costo_trip(new_df.operator[88058][0], new_df.point_timestamp[88058][0], new_df.point_timestamp[88058][-1], Point(new_df.coords[88058][-1][0], new_df.coords[88058][-1][1]))

2.34

### Compute costs for all trips

In [80]:
cost = []
for trip in range(len(new_df)):
    cost.append(costo_trip(new_df.operator[trip][0], new_df.point_timestamp[trip][0], new_df.point_timestamp[trip][-1], Point(new_df.coords[trip][-1][0], new_df.coords[trip][-1][1])))

In [81]:
new_df['costo'] = cost

In [88]:
new_df[new_df.costo>100]

,unique_id,point_timestamp,operator,coords,costo
22888,00000d4b-0000-1000-871b-5b7d6a617b60_2022-02-28,"[2022-02-28 02:09:38+00:00, 2022-02-28 02:15:3...",[BIT],"[[11.129086, 46.055443], [11.124462, 46.048096...",134.880
58370,000021cd-0000-1000-93cd-5b7d6a617b60_2022-03-02,"[2022-03-02 07:20:16+00:00, 2022-03-02 07:25:3...",[BIT],"[[11.125502, 46.059563], [11.119207, 46.066158...",110.078
58896,00002218-0000-1000-95a8-5b7d6a617b60_2022-02-22,"[2022-02-22 06:30:50+00:00, 2022-02-22 06:35:3...",[BIT],"[[11.120316, 46.071616], [11.117499, 46.079905...",102.335
76766,722b44dc8cb4_2021-06-18,"[2021-06-18 06:23:27+00:00, 2021-06-18 06:23:3...",[VENTO],"[[11.123912, 46.067899], [11.123871, 46.067882...",131.430
83499,c627db2e6d60_2021-06-05,"[2021-06-05 06:24:45+00:00, 2021-06-05 06:24:5...",[VENTO],"[[11.115805, 46.081687], [11.115814, 46.081679...",181.350


In [84]:
new_df.to_parquet('trips_cost.parquet')